In [9]:
import json, warnings
from pathlib import Path

import numpy as np
import pandas as pd
#import matplotlib.pyplot as plt
#import seaborn as sns

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.experimental import enable_iterative_imputer  # noqa
from sklearn.impute import IterativeImputer
from imblearn.over_sampling import SMOTENC
from imblearn.under_sampling import RandomUnderSampler

In [10]:
RAW_PATH = "../Dataset/clinical_data.csv"      
OUT_DIR = Path("../Dataset/processed")          # output folder
SEED = 42
N_FOLDS = 5                                     # CV folds inside the training part
TEST_SPLITS = 5                                 # 1/5 of patients -> ~20 % test
RESAMPLE_TARGET = "median"                      # class imbalance handling
                                                
RARE_BACKGROUND_TOP_K = 5                       # categorical feature handling

MALIGNANT = ["BCC", "SCC", "MEL"]
LABEL_MAP = {"NEV": 0, "ACK": 1, "SEK": 2, "BCC": 3, "SCC": 4, "MEL": 5}

OUT_DIR.mkdir(parents=True, exist_ok=True)
json.dump(LABEL_MAP, open(OUT_DIR / "label_map.json", "w"), indent=2)

In [11]:
BOOL_COLS = ["smoke", "drink", "pesticide", "skin_cancer_history", "cancer_history", "has_piped_water",
             "has_sewage_system", "itch", "grew", "hurt", "changed", "bleed", "elevation", "biopsed"]

raw = pd.read_csv(RAW_PATH)
df = raw.copy()
df = df.replace({"UNK": np.nan, "unk": np.nan, "": np.nan})
df = df.rename(columns={"fitspatrick": "fitzpatrick"})

for c in BOOL_COLS:
    df[c] = (df[c].astype(str).str.upper()
               .map({"TRUE": 1, "FALSE": 0, "1": 1, "0": 0, "1.0": 1, "0.0": 0}))
for c in ["age", "fitzpatrick", "diameter_1", "diameter_2"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# invalid diameters: <=0 is impossible -> NaN ; make diameter_1 the longer axis
for c in ["diameter_1", "diameter_2"]:
    df.loc[df[c] <= 0, c] = np.nan
swap = df["diameter_2"] > df["diameter_1"]
print("Rows with diameter_2 > diameter_1 swapped:", int(swap.sum()))
df.loc[swap, ["diameter_1", "diameter_2"]] = df.loc[swap, ["diameter_2", "diameter_1"]].values

# targets
df["is_cancer"] = df["diagnostic"].isin(MALIGNANT).astype(int)
df["y_multiclass"] = df["diagnostic"].map(LABEL_MAP)
assert df["y_multiclass"].notna().all(), "Unknown diagnostic label found"

# identifiers / flags (meta columns, never used as features)
df["meta_patient_id"] = df["patient_id"]
df["meta_lesion_key"] = df["patient_id"].astype(str) + "_" + df["lesion_id"].astype(str)   # lesion_id alone is NOT unique
df["meta_img_id"] = df["img_id"]
df["has_questionnaire"] = (df["gender"].notna() & df["fitzpatrick"].notna() & df["diameter_1"].notna()).astype(int)
df["meta_has_questionnaire"] = df["has_questionnaire"]
df["q_present"] = df["has_questionnaire"]          # usable as an (optional) feature

# engineered features (row-wise, no leakage)
df["grew_unknown"] = df["grew"].isna().astype(int)
df["changed_unknown"] = df["changed"].isna().astype(int)
df["symptom_count"] = df[["itch", "hurt", "bleed", "grew", "changed"]].fillna(0).sum(axis=1)
df["mean_diameter"] = df[["diameter_1", "diameter_2"]].mean(axis=1)
df["diameter_ratio"] = df["diameter_2"] / df["diameter_1"]
df["log_area"] = np.log1p(df["diameter_1"] * df["diameter_2"])

# collapse sparse ancestry labels (frequency only, no label information used)
for c in ["background_father", "background_mother"]:
    top = df[c].value_counts().head(RARE_BACKGROUND_TOP_K).index
    df[c] = df[c].where(df[c].isin(top) | df[c].isna(), "OTHER")

# "UNK" ancestry among rows that DID answer the questionnaire is a genuine answer -> own category
for c in ["background_father", "background_mother"]:
    df.loc[(df["has_questionnaire"] == 1) & df[c].isna(), c] = "UNKNOWN"

df = df.reset_index(drop=True)
df.to_csv(OUT_DIR / "base_cleaned.csv", index=False)
print("Base cleaned shape:", df.shape, "| rows with questionnaire:", int(df.has_questionnaire.sum()))
df[["diagnostic", "y_multiclass", "is_cancer"]].value_counts().sort_index()

Rows with diameter_2 > diameter_1 swapped: 17
Base cleaned shape: (2298, 40) | rows with questionnaire: 1490


diagnostic  y_multiclass  is_cancer
ACK         1             0            730
BCC         3             1            845
MEL         5             1             52
NEV         0             0            244
SCC         4             1            192
SEK         2             0            235
Name: count, dtype: int64

In [12]:
y_all, groups_all = df["diagnostic"], df["patient_id"]

#Outer split train vs test
outer = StratifiedGroupKFold(n_splits=TEST_SPLITS, shuffle=True, random_state=SEED)
train_idx, test_idx = next(outer.split(df, y_all, groups_all))
train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()

#Inner split Cross-validation folds within training
inner = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
train_df["fold"] = -1
for k, (_, val_i) in enumerate(inner.split(train_df, train_df["diagnostic"], train_df["patient_id"])):
    train_df.iloc[val_i, train_df.columns.get_loc("fold")] = k

#Sanity checks
assert not set(train_df.patient_id) & set(test_df.patient_id), "patient overlap train/test!"
for k in range(N_FOLDS):
    assert not (set(train_df[train_df.fold == k].patient_id) & set(train_df[train_df.fold != k].patient_id))

#Save the split assignments
splits = pd.concat([
    train_df[["patient_id", "img_id", "fold"]].assign(split="train"),
    test_df[["patient_id", "img_id"]].assign(split="test", fold=-1)
]).sort_index()
splits.to_csv(OUT_DIR / "splits.csv", index=False)

#Class distribution summary
print(f"train rows {len(train_df)} | test rows {len(test_df)}")
pd.DataFrame({
    "train_%": (train_df.diagnostic.value_counts(normalize=True) * 100).round(1),
    "test_%": (test_df.diagnostic.value_counts(normalize=True) * 100).round(1),
    "train_n": train_df.diagnostic.value_counts(),
    "test_n": test_df.diagnostic.value_counts(),
})

train rows 1838 | test rows 460


,train_%,test_%,train_n,test_n
diagnostic,,,,
ACK,31.9,31.3,586,144
BCC,36.6,37.6,672,173
MEL,2.1,2.8,39,13
NEV,10.8,9.8,199,45
SCC,8.4,8.0,155,37
SEK,10.2,10.4,187,48


In [13]:
FEATURES = {
    "A": dict(
        num=["age", "symptom_count"],
        bin=["itch", "grew", "hurt", "changed", "bleed", "elevation", "grew_unknown", "changed_unknown"],
        cat=["region"]),
    "B": dict(
        num=["age", "symptom_count", "fitzpatrick", "diameter_1", "diameter_2",
             "mean_diameter", "diameter_ratio", "log_area"],
        bin=["itch", "grew", "hurt", "changed", "bleed", "elevation", "grew_unknown", "changed_unknown",
             "smoke", "drink", "pesticide", "skin_cancer_history", "cancer_history",
             "has_piped_water", "has_sewage_system"],
        cat=["region", "gender", "background_father", "background_mother"]),
}

VERSIONS = {
    # Feature set A 
    "a01_A_impute_simple":        dict(fs="A", rows="all",          impute="simple",    flag=False, scale=True,  resample=None),
    "a02_A_dropna":               dict(fs="A", rows="drop_missing", impute="simple",    flag=False, scale=True,  resample=None),
    "a03_A_impute_smote":         dict(fs="A", rows="all",          impute="simple",    flag=False, scale=True,  resample="smote"),
    "a04_A_impute_undersample":   dict(fs="A", rows="all",          impute="simple",    flag=False, scale=True,  resample="under"),
    "a05_A_native_nan":           dict(fs="A", rows="all",          impute="none",      flag=False, scale=False, resample=None),
    # Feature set B 
    "b01_B_questionnaire_only":   dict(fs="B", rows="questionnaire_only", impute="simple", flag=False, scale=True, resample=None),
    "b02_B_impute_simple_flag":   dict(fs="B", rows="all",          impute="simple",    flag=True,  scale=True,  resample=None),
    "b03_B_impute_knn_flag":      dict(fs="B", rows="all",          impute="knn",       flag=True,  scale=True,  resample=None),
    "b04_B_impute_iter_flag":     dict(fs="B", rows="all",          impute="iterative", flag=True,  scale=True,  resample=None),
    "b05_B_quest_only_smote":     dict(fs="B", rows="questionnaire_only", impute="simple", flag=False, scale=True, resample="smote"),
    "b06_B_impute_flag_smote":    dict(fs="B", rows="all",          impute="simple",    flag=True,  scale=True,  resample="smote"),
    "b07_B_native_nan_flag":      dict(fs="B", rows="all",          impute="none",      flag=True,  scale=False, resample=None),
    "b08_B_strict_complete_case": dict(fs="B", rows="drop_missing", impute="simple",    flag=False, scale=True,  resample=None),
}
print(len(VERSIONS), "versions defined")

13 versions defined


In [ ]:
class Prep:
    def __init__(self, cfg):
        self.cfg = cfg
        fs = FEATURES[cfg["fs"]]
        self.num, self.bin, self.cat = list(fs["num"]), list(fs["bin"]), list(fs["cat"])
        if cfg["flag"]:
            self.bin.append("q_present")
        self.nb = self.num + self.bin

    # ---- fit 
    def fit(self, tr):
        mode = self.cfg["impute"]
        X = tr[self.nb].astype(float)
        self.lo, self.hi = X.min(), X.max()
        if mode == "simple":
            self.imp_num = SimpleImputer(strategy="median", keep_empty_features=True).fit(X[self.num])
            self.imp_bin = SimpleImputer(strategy="most_frequent", keep_empty_features=True).fit(X[self.bin])
        elif mode == "knn":
            self.pre = StandardScaler().fit(X)                      # distances need comparable scales
            self.imp = KNNImputer(n_neighbors=5, weights="distance", keep_empty_features=True).fit(self.pre.transform(X))
        elif mode == "iterative":
            self.imp = IterativeImputer(max_iter=10, random_state=SEED, initial_strategy="median",
                                        keep_empty_features=True).fit(X)
        if mode != "none":
            self.imp_cat = SimpleImputer(strategy="most_frequent", keep_empty_features=True).fit(tr[self.cat])
        self.ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False, drop="if_binary").fit(self._cat(tr))
        if self.cfg["scale"]:
            self.scaler = StandardScaler().fit(self._nb(tr)[self.num])
        return self

    # ---- helpers
    def _nb(self, d):
        mode = self.cfg["impute"]
        X = d[self.nb].astype(float)
        if mode == "simple":
            n = pd.DataFrame(self.imp_num.transform(X[self.num]), columns=self.num, index=d.index)
            b = pd.DataFrame(self.imp_bin.transform(X[self.bin]), columns=self.bin, index=d.index)
            return pd.concat([n, b], axis=1)
        if mode == "knn":
            Z = self.imp.transform(self.pre.transform(X))
            X = pd.DataFrame(self.pre.inverse_transform(Z), columns=self.nb, index=d.index)
        elif mode == "iterative":
            X = pd.DataFrame(self.imp.transform(X), columns=self.nb, index=d.index)
        if mode in ("knn", "iterative"):
            X[self.bin] = X[self.bin].round().clip(0, 1)                     # binary stays binary
            X = X.clip(lower=self.lo, upper=self.hi, axis=1)                 # stay inside the training range
        return X

    def _cat(self, d):
        c = d[self.cat].copy()
        if self.cfg["impute"] == "none":
            return c.fillna("MISSING")
        return pd.DataFrame(self.imp_cat.transform(c), columns=self.cat, index=d.index)

    # ---- transform
    def transform(self, d):
        Xnb = self._nb(d)
        if self.cfg["scale"]:
            Xnb[self.num] = self.scaler.transform(Xnb[self.num])
        oh = pd.DataFrame(self.ohe.transform(self._cat(d)), columns=self.ohe.get_feature_names_out(self.cat), index=d.index)
        return pd.concat([Xnb, oh], axis=1)


TARGET_COLS = ["diagnostic", "y_multiclass", "is_cancer"]
META_COLS = ["meta_patient_id", "meta_lesion_key", "meta_img_id", "meta_has_questionnaire"]


def target_frame(y):
    return pd.DataFrame({"diagnostic": y.values, "y_multiclass": y.map(LABEL_MAP).values,
                         "is_cancer": y.isin(MALIGNANT).astype(int).values}, index=y.index)


def resample_train(feat, meta, y, cfg, num_cols_kept):
    """Resample TRAINING data only. Returns (feat, meta, y) with meta_is_synthetic added."""
    counts = y.value_counts()
    target = int(counts.median()) if RESAMPLE_TARGET == "median" else int(counts.max())
    meta = meta.copy()
    meta["meta_is_synthetic"] = 0

    if cfg["resample"] == "under":
        strategy = {k: target for k, v in counts.items() if v > target}
        if not strategy:
            return feat, meta, y
        rus = RandomUnderSampler(sampling_strategy=strategy, random_state=SEED)
        rus.fit_resample(feat, y)
        keep = np.sort(rus.sample_indices_)                 # keep original row order
        return feat.iloc[keep], meta.iloc[keep], y.iloc[keep]

    if cfg["resample"] == "smote":
        strategy = {k: target for k, v in counts.items() if v < target}
        if not strategy:
            return feat, meta, y
        cat_idx = [i for i, c in enumerate(feat.columns) if c not in num_cols_kept]
        k = int(max(1, min(5, counts.min() - 1)))
        sm = SMOTENC(categorical_features=cat_idx, sampling_strategy=strategy, k_neighbors=k, random_state=SEED)
        Xr, yr = sm.fit_resample(feat.reset_index(drop=True), y.reset_index(drop=True))
        n0 = len(feat)
        assert np.allclose(Xr.iloc[:n0].values.astype(float), feat.values.astype(float), equal_nan=True), "row order changed"
        syn = len(Xr) - n0
        meta_syn = pd.DataFrame({c: np.nan for c in meta.columns}, index=range(syn))
        meta_syn["meta_patient_id"] = "SYNTHETIC"
        meta_syn["meta_is_synthetic"] = 1
        meta_all = pd.concat([meta.reset_index(drop=True), meta_syn], ignore_index=True)
        return Xr, meta_all, yr
    return feat, meta, y


def build(cfg, tr, others):
    """Fit on `tr`, return processed train + dict of processed other frames."""
    tr = tr.copy()
    n_before = len(tr)
    if cfg["rows"] == "questionnaire_only":
        tr = tr[tr["has_questionnaire"] == 1]
    elif cfg["rows"] == "drop_missing":
        p = Prep(cfg)
        tr = tr.dropna(subset=[c for c in p.num + p.bin + p.cat if c != "q_present"])
    prep = Prep(cfg).fit(tr)

    feat_tr = prep.transform(tr)
    const = [c for c in feat_tr.columns if feat_tr[c].nunique(dropna=False) <= 1]   # useless in this version
    keep_cols = [c for c in feat_tr.columns if c not in const]
    feat_tr = feat_tr[keep_cols]
    num_kept = [c for c in prep.num if c in keep_cols]

    meta_tr = tr[META_COLS].copy()
    y_tr = tr["diagnostic"]
    if cfg["resample"]:
        feat_tr, meta_tr, y_tr = resample_train(feat_tr, meta_tr, y_tr, cfg, num_kept)
    else:
        meta_tr["meta_is_synthetic"] = 0

    def pack(feat, meta, y):
        return pd.concat([meta.reset_index(drop=True), target_frame(y.reset_index(drop=True)),
                          feat.reset_index(drop=True)], axis=1)

    out_tr = pack(feat_tr, meta_tr, y_tr)
    outs = {}
    for name, d in others.items():
        m = d[META_COLS].copy(); m["meta_is_synthetic"] = 0
        outs[name] = pack(prep.transform(d)[keep_cols], m, d["diagnostic"])
    info = dict(rows_dropped=n_before - len(tr), constant_cols_removed=const,
                numeric_cols=num_kept, feature_cols=keep_cols)
    return out_tr, outs, info

In [15]:
final = []
for name, cfg in VERSIONS.items():
    vdir = OUT_DIR / name
    vdir.mkdir(exist_ok=True)

    # (a) final train/test
    tr_out, outs, info = build(cfg, train_df, {"test": test_df})
    tr_out.to_csv(vdir / "final_train.csv", index=False)
    outs["test"].to_csv(vdir / "final_test.csv", index=False)
    json.dump({"features": info["feature_cols"], "numeric": info["numeric_cols"], "config": cfg,
               "meta_cols": META_COLS + ["meta_is_synthetic"], "target_cols": TARGET_COLS},
              open(vdir / "feature_columns.json", "w"), indent=2)

    # (b) CV folds
    for k in range(N_FOLDS):
        f_tr, f_out, _ = build(cfg, train_df[train_df.fold != k], {"val": train_df[train_df.fold == k]})
        f_tr.to_csv(vdir / f"fold_{k}_train.csv", index=False)
        f_out["val"].to_csv(vdir / f"fold_{k}_val.csv", index=False)

    vc = tr_out["diagnostic"].value_counts()
    final.append(dict(version=name, **cfg,
                         train_rows=len(tr_out), synthetic_rows=int(tr_out["meta_is_synthetic"].sum()),
                         rows_dropped_from_train=info["rows_dropped"], test_rows=len(outs["test"]),
                         n_features=len(info["feature_cols"]),
                         nan_in_train=int(tr_out[info["feature_cols"]].isna().sum().sum()),
                         **{f"n_{c}": int(vc.get(c, 0)) for c in LABEL_MAP}))
    print(f"saved {name:28s} train={len(tr_out):5d}  features={len(info['feature_cols']):3d}  "
          f"dropped={info['rows_dropped']:4d}  synthetic={int(tr_out['meta_is_synthetic'].sum())}")

final = pd.DataFrame(final)
final.to_csv(OUT_DIR / "final.csv", index=False)
final

saved a01_A_impute_simple          train= 1838  features= 24  dropped=   0  synthetic=0
saved a02_A_dropna                 train= 1521  features= 22  dropped= 317  synthetic=0


c:\ProgramData\anaconda3\envs\notebook_env\lib\site-packages\sklearn\base.py:484: FutureWarning: `BaseEstimator._check_n_features` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation._check_n_features` instead.
  warnings.warn(
c:\ProgramData\anaconda3\envs\notebook_env\lib\site-packages\sklearn\base.py:493: FutureWarning: `BaseEstimator._check_feature_names` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation._check_feature_names` instead.
  warnings.warn(
c:\ProgramData\anaconda3\envs\notebook_env\lib\site-packages\sklearn\base.py:484: FutureWarning: `BaseEstimator._check_n_features` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation._check_n_features` instead.
  warnings.warn(
c:\ProgramData\anaconda3\envs\notebook_env\lib\site-packages\sklearn\base.py:493: FutureWarning: `BaseEstimator._check_feature_names` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation._check_feature_n

saved a03_A_impute_smote           train= 2036  features= 24  dropped=   0  synthetic=198


c:\ProgramData\anaconda3\envs\notebook_env\lib\site-packages\sklearn\base.py:484: FutureWarning: `BaseEstimator._check_n_features` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation._check_n_features` instead.
  warnings.warn(
c:\ProgramData\anaconda3\envs\notebook_env\lib\site-packages\sklearn\base.py:493: FutureWarning: `BaseEstimator._check_feature_names` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation._check_feature_names` instead.
  warnings.warn(
c:\ProgramData\anaconda3\envs\notebook_env\lib\site-packages\sklearn\base.py:484: FutureWarning: `BaseEstimator._check_n_features` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation._check_n_features` instead.
  warnings.warn(
c:\ProgramData\anaconda3\envs\notebook_env\lib\site-packages\sklearn\base.py:493: FutureWarning: `BaseEstimator._check_feature_names` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation._check_feature_n

saved a04_A_impute_undersample     train=  960  features= 24  dropped=   0  synthetic=0
saved a05_A_native_nan             train= 1838  features= 24  dropped=   0  synthetic=0
saved b01_B_questionnaire_only     train= 1189  features= 52  dropped= 649  synthetic=0
saved b02_B_impute_simple_flag     train= 1838  features= 53  dropped=   0  synthetic=0
saved b03_B_impute_knn_flag        train= 1838  features= 53  dropped=   0  synthetic=0
saved b04_B_impute_iter_flag       train= 1838  features= 53  dropped=   0  synthetic=0


c:\ProgramData\anaconda3\envs\notebook_env\lib\site-packages\sklearn\base.py:484: FutureWarning: `BaseEstimator._check_n_features` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation._check_n_features` instead.
  warnings.warn(
c:\ProgramData\anaconda3\envs\notebook_env\lib\site-packages\sklearn\base.py:493: FutureWarning: `BaseEstimator._check_feature_names` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation._check_feature_names` instead.
  warnings.warn(
c:\ProgramData\anaconda3\envs\notebook_env\lib\site-packages\sklearn\base.py:484: FutureWarning: `BaseEstimator._check_n_features` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation._check_n_features` instead.
  warnings.warn(
c:\ProgramData\anaconda3\envs\notebook_env\lib\site-packages\sklearn\base.py:493: FutureWarning: `BaseEstimator._check_feature_names` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation._check_feature_n

saved b05_B_quest_only_smote       train= 1375  features= 52  dropped= 649  synthetic=186


c:\ProgramData\anaconda3\envs\notebook_env\lib\site-packages\sklearn\base.py:484: FutureWarning: `BaseEstimator._check_n_features` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation._check_n_features` instead.
  warnings.warn(
c:\ProgramData\anaconda3\envs\notebook_env\lib\site-packages\sklearn\base.py:493: FutureWarning: `BaseEstimator._check_feature_names` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation._check_feature_names` instead.
  warnings.warn(
c:\ProgramData\anaconda3\envs\notebook_env\lib\site-packages\sklearn\base.py:484: FutureWarning: `BaseEstimator._check_n_features` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation._check_n_features` instead.
  warnings.warn(
c:\ProgramData\anaconda3\envs\notebook_env\lib\site-packages\sklearn\base.py:493: FutureWarning: `BaseEstimator._check_feature_names` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation._check_feature_n

saved b06_B_impute_flag_smote      train= 2036  features= 53  dropped=   0  synthetic=198
saved b07_B_native_nan_flag        train= 1838  features= 57  dropped=   0  synthetic=0
saved b08_B_strict_complete_case   train=  873  features= 50  dropped= 965  synthetic=0


,version,fs,rows,impute,flag,scale,resample,train_rows,synthetic_rows,rows_dropped_from_train,test_rows,n_features,nan_in_train,n_NEV,n_ACK,n_SEK,n_BCC,n_SCC,n_MEL
0,a01_A_impute_simple,A,all,simple,False,True,None,1838,0,0,460,24,0,199,586,187,672,155,39
1,a02_A_dropna,A,drop_missing,simple,False,True,None,1521,0,317,460,22,0,168,582,179,470,92,30
2,a03_A_impute_smote,A,all,simple,False,True,smote,2036,198,0,460,24,0,199,586,193,672,193,193
3,a04_A_impute_undersample,A,all,simple,False,True,under,960,0,0,460,24,0,193,193,187,193,155,39
4,a05_A_native_nan,A,all,none,False,False,None,1838,0,0,460,24,646,199,586,187,672,155,39
5,b01_B_questionnaire_only,B,questionnaire_only,simple,False,True,None,1189,0,649,460,52,0,67,221,38,670,154,39
6,b02_B_impute_simple_flag,B,all,simple,True,True,None,1838,0,0,460,53,0,199,586,187,672,155,39
7,b03_B_impute_knn_flag,B,all,knn,True,True,None,1838,0,0,460,53,0,199,586,187,672,155,39
8,b04_B_impute_iter_flag,B,all,iterative,True,True,None,1838,0,0,460,53,0,199,586,187,672,155,39
9,b05_B_quest_only_smote,B,questionnaire_only,simple,False,True,smote,1375,186,649,460,52,0,110,221,110,670,154,110
